In [31]:
from typing import final

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

#reading data
df_full = pd.read_csv("005_manchester_processed.csv")

#sort
df_full = df_full.sort_values(by='time').reset_index(drop=True)

# choose training and test dataset
df = df_full[(df_full['year'] >= 2006) & (df_full['year'] <= 2050)].copy()


df = df.drop(columns=['lat', 'lon', 'date'], errors='ignore')

print("data shape：", df.shape)
print(df.head())

data shape： (15816, 14)
         time  TREFMXAV_U       FLNS       FSNS         PRECT          PRSN  \
0  2006-01-02    11.15260   8.737288   7.855509  1.544200e-07  4.051599e-16   
1  2006-01-03    13.01327   6.686464   7.501073  7.784098e-08  0.000000e+00   
2  2006-01-04    13.16030  27.445148  12.188718  4.851411e-08  7.075068e-18   
3  2006-01-05    14.85882  10.443632   4.354691  1.091676e-07  1.429017e-14   
4  2006-01-06    10.78576  70.927300  31.532597  2.531008e-09  6.418103e-18   

       QBOT    TREFHT      UBOT      VBOT  year  month  day  dayofyear  
0  0.005373   6.07195  2.303055  4.502803  2006      1    2          2  
1  0.007595  11.04843  4.657475  3.158464  2006      1    3          3  
2  0.005667   9.27023  5.083677  5.835154  2006      1    4          4  
3  0.007362  11.82208  3.029053  8.031938  2006      1    5          5  
4  0.004160   7.14828  3.783906  6.986506  2006      1    6          6  


In [32]:
target = 'TREFMXAV_U'

features = [col for col in df.columns if col not in [target, 'time']]

X_all = df[features].values
y_all = df[target].values

In [33]:

train_years = 5
val_years = 3
stride_years = 5

days_per_year = 365

train_size = train_years * days_per_year
val_size = val_years * days_per_year
stride_size = stride_years * days_per_year


def sliding_split(X, y, df, train_size, val_size, stride_size):
    splits = []

    start = 0
    total = len(X)

    while start + train_size + val_size <= total:

        train_end = start + train_size
        val_end = train_end + val_size

        X_train = X[start:train_end]
        y_train = y[start:train_end]

        X_val = X[train_end:val_end]
        y_val = y[train_end:val_end]

        # record time
        train_start_year = df.iloc[start]['year']
        train_end_year = df.iloc[train_end - 1]['year']

        val_start_year = df.iloc[train_end]['year']
        val_end_year = df.iloc[val_end - 1]['year']

        splits.append((
            X_train, y_train, X_val, y_val,
            train_start_year, train_end_year,
            val_start_year, val_end_year
        ))

        start += stride_size

    return splits


splits = sliding_split(X_all, y_all, df, train_size, val_size, stride_size)

print("Windows：", len(splits))

Windows： 8


In [34]:
mae_list = []
rmse_list = []

for i, (X_train, y_train, X_val, y_val,
        train_start_year, train_end_year,
        val_start_year, val_end_year) in enumerate(splits):

    print(f"\n===== Split {i+1} =====")
    print(f"Train: {train_start_year} → {train_end_year}")
    print(f"Val  : {val_start_year} → {val_end_year}")
    print(f"Train size: {len(X_train)} | Val size: {len(X_val)}")

    rf = RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)

    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    mae_list.append(mae)
    rmse_list.append(rmse)

    print(f"MAE: {mae:.3f}, RMSE: {rmse:.3f}")

print("\n===== result =====")
print("Average MAE:", np.mean(mae_list))
print("Average RMSE:", np.mean(rmse_list))


===== Split 1 =====
Train: 2006 → 2011
Val  : 2011 → 2014
Train size: 1825 | Val size: 1095
MAE: 0.560, RMSE: 0.745

===== Split 2 =====
Train: 2011 → 2016
Val  : 2016 → 2019
Train size: 1825 | Val size: 1095
MAE: 0.585, RMSE: 0.786

===== Split 3 =====
Train: 2016 → 2021
Val  : 2021 → 2024
Train size: 1825 | Val size: 1095
MAE: 0.602, RMSE: 0.795

===== Split 4 =====
Train: 2021 → 2026
Val  : 2026 → 2029
Train size: 1825 | Val size: 1095
MAE: 0.588, RMSE: 0.786

===== Split 5 =====
Train: 2026 → 2031
Val  : 2031 → 2035
Train size: 1825 | Val size: 1095
MAE: 0.593, RMSE: 0.770

===== Split 6 =====
Train: 2031 → 2037
Val  : 2037 → 2040
Train size: 1825 | Val size: 1095
MAE: 0.601, RMSE: 0.773

===== Split 7 =====
Train: 2037 → 2042
Val  : 2042 → 2045
Train size: 1825 | Val size: 1095
MAE: 0.627, RMSE: 0.839

===== Split 8 =====
Train: 2042 → 2047
Val  : 2047 → 2050
Train size: 1825 | Val size: 1095
MAE: 0.593, RMSE: 0.796

===== result =====
Average MAE: 0.5937027113966902
Average RMSE

In [35]:
final_train_df = df_full[df_full['year'] <= 2050]
final_test_df  = df_full[df_full['year'] > 2050]
X_final_train = final_train_df[features].values
y_final_train = final_train_df[target].values

X_final_test = final_test_df[features].values
y_final_test = final_test_df[target].values


In [36]:
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_final_train, y_final_train)

RandomForestRegressor(max_depth=10, n_jobs=-1, random_state=42)

In [37]:
from sklearn.metrics import r2_score, max_error

y_pred = rf.predict(X_final_test)

rf_mae = mean_absolute_error(y_final_test, y_pred)
rf_rmse = np.sqrt(mean_squared_error(y_final_test, y_pred))
rf_r2 = r2_score(y_final_test, y_pred)
rf_max_err = max_error(y_final_test, y_pred)

print("===== Test Results =====")
print(f"MAE: {rf_mae:.3f}")
print(f"RMSE: {rf_rmse:.3f}")
print(f"R2: {rf_r2:.3f}")
print(f"Max Error: {rf_max_err:.3f}")

===== Test Results =====
MAE: 0.564
RMSE: 0.735
R2: 0.981
Max Error: 4.946


In [38]:
from xgboost import XGBRegressor
mae_list_xgb = []
rmse_list_xgb = []

for i, (X_train, y_train, X_val, y_val,
        train_start_year, train_end_year,
        val_start_year, val_end_year) in enumerate(splits):

    print(f"\n===== XGB Split {i+1} =====")
    print(f"Train: {train_start_year} → {train_end_year}")
    print(f"Val  : {val_start_year} → {val_end_year}")

    xgb = XGBRegressor(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    xgb.fit(X_train, y_train)

    y_pred = xgb.predict(X_val)

    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))


    mae_list_xgb.append(mae)
    rmse_list_xgb.append(rmse)

    print(f"XGB MAE: {mae:.3f}, RMSE: {rmse:.3f}")

print("\n===== XGBoost result =====")
print("Average MAE:", np.mean(mae_list_xgb))
print("Average RMSE:", np.mean(rmse_list_xgb))


===== XGB Split 1 =====
Train: 2006 → 2011
Val  : 2011 → 2014
XGB MAE: 0.542, RMSE: 0.710

===== XGB Split 2 =====
Train: 2011 → 2016
Val  : 2016 → 2019
XGB MAE: 0.543, RMSE: 0.751

===== XGB Split 3 =====
Train: 2016 → 2021
Val  : 2021 → 2024
XGB MAE: 0.561, RMSE: 0.747

===== XGB Split 4 =====
Train: 2021 → 2026
Val  : 2026 → 2029
XGB MAE: 0.563, RMSE: 0.746

===== XGB Split 5 =====
Train: 2026 → 2031
Val  : 2031 → 2035
XGB MAE: 0.575, RMSE: 0.751

===== XGB Split 6 =====
Train: 2031 → 2037
Val  : 2037 → 2040
XGB MAE: 0.561, RMSE: 0.733

===== XGB Split 7 =====
Train: 2037 → 2042
Val  : 2042 → 2045
XGB MAE: 0.581, RMSE: 0.787

===== XGB Split 8 =====
Train: 2042 → 2047
Val  : 2047 → 2050
XGB MAE: 0.561, RMSE: 0.764

===== XGBoost result =====
Average MAE: 0.5610000992543973
Average RMSE: 0.7486611615063057


In [39]:
xgb_final = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_final.fit(X_final_train, y_final_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=-1, num_parallel_tree=None, ...)

In [40]:
from sklearn.metrics import r2_score, max_error

y_pred = xgb_final.predict(X_final_test)

xgb_mae = mean_absolute_error(y_final_test, y_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_final_test, y_pred))
xgb_r2 = r2_score(y_final_test, y_pred)
xgb_max_err = max_error(y_final_test, y_pred)

print("\n===== XGBoost Test Results =====")
print(f"MAE: {xgb_mae:.3f}")
print(f"RMSE: {xgb_rmse:.3f}")
print(f"R2: {xgb_r2:.3f}")
print(f"Max Error: {xgb_max_err:.3f}")


===== XGBoost Test Results =====
MAE: 0.519
RMSE: 0.687
R2: 0.984
Max Error: 4.283


In [41]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae_list_lgb = []
rmse_list_lgb = []

for i, (X_train, y_train, X_val, y_val,
        train_start_year, train_end_year,
        val_start_year, val_end_year) in enumerate(splits):

    print(f"\n===== LGBM Split {i+1} =====")
    print(f"Train: {train_start_year} → {train_end_year}")
    print(f"Val  : {val_start_year} → {val_end_year}")

    lgb = LGBMRegressor(
        n_estimators=300,
        max_depth=-1,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )


    lgb.fit(X_train, y_train)


    y_pred = lgb.predict(X_val)


    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    mae_list_lgb.append(mae)
    rmse_list_lgb.append(rmse)

    print(f"LGBM MAE: {mae:.3f}, RMSE: {rmse:.3f}")


print("\n===== LightGBM Result =====")
print("Average MAE:", np.mean(mae_list_lgb))
print("Average RMSE:", np.mean(rmse_list_lgb))


===== LGBM Split 1 =====
Train: 2006 → 2011
Val  : 2011 → 2014
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000138 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 1825, number of used features: 12
[LightGBM] [Info] Start training from score 13.863594
LGBM MAE: 0.555, RMSE: 0.720

===== LGBM Split 2 =====
Train: 2011 → 2016
Val  : 2016 → 2019
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000107 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2548
[LightGBM] [Info] Number of data points in the train set: 1825, number of used features: 12
[LightGBM] [Info] Start training from score 14.515770


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


LGBM MAE: 0.544, RMSE: 0.736

===== LGBM Split 3 =====
Train: 2016 → 2021
Val  : 2021 → 2024
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000106 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2546
[LightGBM] [Info] Number of data points in the train set: 1825, number of used features: 12
[LightGBM] [Info] Start training from score 14.613395
LGBM MAE: 0.568, RMSE: 0.748

===== LGBM Split 4 =====
Train: 2021 → 2026
Val  : 2026 → 2029
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000092 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2554
[LightGBM] [Info] Number of data points in the train set: 1825, number of used features: 12
[LightGBM] [Info] Start training from score 15.492725


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


LGBM MAE: 0.544, RMSE: 0.732

===== LGBM Split 5 =====
Train: 2026 → 2031
Val  : 2031 → 2035
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000104 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2540
[LightGBM] [Info] Number of data points in the train set: 1825, number of used features: 12
[LightGBM] [Info] Start training from score 14.969676


...

/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


LGBM MAE: 0.573, RMSE: 0.754

===== LGBM Split 7 =====
Train: 2037 → 2042
Val  : 2042 → 2045
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000152 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2544
[LightGBM] [Info] Number of data points in the train set: 1825, number of used features: 12
[LightGBM] [Info] Start training from score 15.147658


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


LGBM MAE: 0.592, RMSE: 0.802

===== LGBM Split 8 =====
Train: 2042 → 2047
Val  : 2047 → 2050
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000129 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2548
[LightGBM] [Info] Number of data points in the train set: 1825, number of used features: 12
[LightGBM] [Info] Start training from score 16.051365
LGBM MAE: 0.573, RMSE: 0.780

===== LightGBM Result =====
Average MAE: 0.5650751429294012
Average RMSE: 0.7520638613997379


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [42]:
lgb_final = LGBMRegressor(
        n_estimators=300,
        max_depth=-1,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )


lgb_final.fit(X_final_train, y_final_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000221 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2593
[LightGBM] [Info] Number of data points in the train set: 15816, number of used features: 12
[LightGBM] [Info] Start training from score 15.026336


LGBMRegressor(colsample_bytree=0.8, learning_rate=0.05, n_estimators=300,
              n_jobs=-1, random_state=42, subsample=0.8)

In [43]:
from sklearn.metrics import r2_score, max_error

y_pred = lgb_final.predict(X_final_test)

lgb_mae = mean_absolute_error(y_final_test, y_pred)
lgb_rmse = np.sqrt(mean_squared_error(y_final_test, y_pred))
lgb_r2 = r2_score(y_final_test, y_pred)
lgb_max_err = max_error(y_final_test, y_pred)

print("\n===== LightGBM Test Results =====")
print(f"MAE: {lgb_mae:.3f}")
print(f"RMSE: {lgb_rmse:.3f}")
print(f"R2: {lgb_r2:.3f}")
print(f"Max Error: {lgb_max_err:.3f}")


===== LightGBM Test Results =====
MAE: 0.516
RMSE: 0.681
R2: 0.984
Max Error: 4.403


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
